# Notebook 02 - Texas energy infrastructure

In [1]:
# I import Path to manage file and folder locations
# without manually combining directory strings.
from pathlib import Path

# I import os for operating-system operations,
# json for reading and writing metadata,
# hashlib for checking whether input files have changed,
# time for tracking execution time,
# and warnings for handling Python warning messages.
import os, json, hashlib, time, warnings

# I import pandas to work with tabular datasets.
import pandas as pd

# I import GeoPandas to read, process, and save spatial datasets.
import geopandas as gpd

# I import NumPy for numerical calculations and array operations.
import numpy as np

# I import tqdm to display progress bars during long operations.
from tqdm.auto import tqdm

# I import unary_union to combine multiple geometries
# into a single geometric representation when needed.
from shapely.ops import unary_union

# I import display to show tables and other outputs
# clearly inside my Jupyter notebook.
from IPython.display import display


# ---------------------------------------------------------
# I define the project directories and processing settings.
# ---------------------------------------------------------

# I define the main folder containing all project files.
# I use the same project root across notebooks so that
# the data checker can follow a consistent folder structure.
PROJECT_ROOT = Path(
    r"G:/My Drive/Projects/Large projects/2026TexasEnergyFootprint"
)

# I define the folder containing cleaned datasets
# exported from Notebook 01.
# I use these files as inputs for the spatial analysis.
CLEAN = PROJECT_ROOT / "03_Cleaned_Data"

# I define the folder containing the original raw datasets.
# I keep raw data separate from processed data
# to preserve the original source files.
RAW = PROJECT_ROOT / "01_Raw_Data"

# I define the folder where I save the spatial analysis
# outputs generated by Notebook 02.
OUT = PROJECT_ROOT / "04_Spatial_Analysis"

# I create the output folder if it does not already exist.
# I use exist_ok=True to avoid an error if the folder exists.
OUT.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# I define settings that control selected processing steps.
# ---------------------------------------------------------

# I set this to False to allow the pipeline analysis
# to reuse an existing county-clipped file when its
# cache validation checks pass.
# If the cache is missing or fails validation,
# the later code rebuilds the county intersection.
# I can set this to True to force a new calculation.
REBUILD_PIPELINE_CLIP = False

# I set this to True to request fresh UIC data
# from the API in the later processing section.
# This means I do not rely only on a previously saved
# local copy of the UIC data.
RELOAD_UIC_API = True

# I set this to True to enable the refinery API
# processing section later in the notebook.
# I use this step to retrieve refinery location data.
RUN_REFINERY_API = True


# ---------------------------------------------------------
# I define unit conversion constants.
# ---------------------------------------------------------

# I define the conversion factor from square meters to acres.
# One acre equals approximately 4,046.8564224 square meters.
# I multiply an area in square meters by this factor
# when I need to report land area in acres.
ACRES_PER_SQM = 1 / 4046.8564224

# I define the number of meters in one mile.
# I divide pipeline lengths in meters by this value
# when I calculate pipeline mileage.
M_PER_MILE = 1609.344


# ---------------------------------------------------------
# I define a function for saving GeoPackage files.
# ---------------------------------------------------------

# I create a reusable function to save spatial datasets.
# I provide the GeoDataFrame, output file path,
# and GeoPackage layer name as inputs.
def save_gpkg(frame, path, layer):

    # I convert the supplied file path to a Path object
    # so I can use consistent file-management operations.
    path = Path(path)

    # I create the parent folder if it does not exist.
    # This prevents an error when saving to a new directory.
    path.parent.mkdir(parents=True, exist_ok=True)

    # I create a copy of the input GeoDataFrame
    # so I do not modify the original dataset.
    # I also reset its index to a sequential row index.
    data = frame.copy().reset_index(drop=True)

    # I search for columns named FID, regardless of capitalization.
    # GeoPackage uses FID as a feature identifier,
    # so an existing column with that name could conflict
    # with the identifier created by the output format.
    fid_cols = [
        c for c in data.columns
        if c.lower() == "fid"
    ]

    # I check whether the input contains an FID column.
    if fid_cols:

        # I confirm that source_fid does not already exist.
        # This prevents me from accidentally overwriting
        # an existing column during the renaming step.
        assert "source_fid" not in data.columns

        # I rename the original FID column to source_fid.
        # This preserves the original identifier as an attribute
        # while allowing GeoPackage to manage its own feature ID.
        data = data.rename(
            columns={fid_cols[0]: "source_fid"}
        )

    # I check whether a file already exists at the output path.
    if path.exists():

        # I delete the existing output file before saving.
        # This prevents older layers or records from remaining
        # in the file when I regenerate the dataset.
        # I only apply this operation to the specified output path.
        path.unlink()

    # I save the GeoDataFrame as a GeoPackage file.
    # I use the specified layer name and do not export
    # the pandas index as an additional attribute column.
    data.to_file(
        path,
        layer=layer,
        driver="GPKG",
        index=False
    )

    # I print the output filename and number of saved records.
    # This helps me confirm that the export was completed
    # and compare the record count with earlier processing steps.
    print(
        f"I saved {path.name}: {len(data):,} rows"
    )


# ---------------------------------------------------------
# I define a function for reading GeoPackage files.
# ---------------------------------------------------------

# I create a reusable function to load a specific layer
# from a GeoPackage file.
def read_gpkg(path, layer):

    # I check whether the required input file exists.
    if not path.exists():

        # I stop execution if the file is missing.
        # I explain that the expected input should have
        # been generated by Notebook 01.
        # This prevents the analysis from continuing
        # with missing or unintended input data.
        raise FileNotFoundError(
            f"I need to rerun Notebook 01: {path}"
        )

    # I read the specified GeoPackage layer
    # and return it as a GeoDataFrame for spatial analysis.
    return gpd.read_file(
        path,
        layer=layer
    )

In [2]:
# I load the cleaned Texas wind turbine locations exported from Notebook 01.
# I use GeoPandas because the GeoJSON contains geographic coordinates.
wind_tx = gpd.read_file(CLEAN / "Texas_Wind_Turbines.geojson")

# I load the cleaned Texas solar project polygons.
# I will use these geometries for subsequent spatial analysis.
solar_tx = gpd.read_file(CLEAN / "Texas_Solar_Projects.geojson")

# I load the cleaned Texas battery storage plant locations.
battery_tx = gpd.read_file(CLEAN / "Texas_Battery_Plants.geojson")

# I load the complete Texas well surface-location inventory.
# I specify the GeoPackage layer created in Notebook 01.
# I retain all well categories rather than loading a restricted subset.
wells_tx = read_gpkg(
    CLEAN / "Texas_Well_Surface_Locations.gpkg",
    "texas_wells"
)

# I load the complete Texas pipeline inventory.
# I retain all original pipeline statuses and system types.
pipelines_all_tx = read_gpkg(
    CLEAN / "Texas_Pipelines_All_Statuses.gpkg",
    "texas_pipelines"
)

# I separately load pipelines classified as in service.
# I use this dataset when the analysis requires in-service infrastructure.
pipelines_active_tx = read_gpkg(
    CLEAN / "Texas_Pipelines_In_Service.gpkg",
    "texas_pipelines"
)


# ---------------------------------------------------------
# I check that the cleaned datasets loaded correctly.
# ---------------------------------------------------------

# I organize the six datasets in a dictionary.
# This allows me to apply the same validation checks
# to each dataset without repeating the code.
for name, frame in {
    "Wind": wind_tx,
    "Solar": solar_tx,
    "Battery": battery_tx,
    "Wells": wells_tx,
    "Pipeline all": pipelines_all_tx,
    "Pipeline in service": pipelines_active_tx
}.items():

    # I verify that each dataset has a defined
    # coordinate reference system (CRS).
    # I need a CRS to correctly transform and compare
    # the geographic locations in subsequent spatial operations.
    assert frame.crs is not None, (
        f"I need a source CRS for {name}"
    )

    # I display the dataset name, record count, CRS,
    # geometry types, and number of missing geometries.
    # I use these diagnostics to identify unexpected
    # changes in the cleaned datasets.
    print(
        name,
        f"{len(frame):,}",
        frame.crs,
        frame.geom_type.value_counts().to_dict(),
        "missing geometry",
        int(frame.geometry.isna().sum())
    )


# ---------------------------------------------------------
# I verify the classification fields required for analysis.
# ---------------------------------------------------------

# I confirm that the well dataset contains the original
# RRC symbol codes and the well categories assigned in Notebook 01.
# I need these fields to examine different well classifications
# without losing the original RRC information.
assert {"SYMNUM", "well_category"}.issubset(
    wells_tx.columns
), "I need the complete classified well export"

# I confirm that the complete pipeline inventory includes
# the RRC status and system-type classification fields.
# I need these fields to distinguish pipeline operating
# statuses and commodities during subsequent analysis.
assert {"STATUS_CD", "SYSTYPE"}.issubset(
    pipelines_all_tx.columns
)

# I verify that every record in the in-service pipeline
# dataset has the RRC status code I.
# I convert values to strings and remove surrounding spaces
# to avoid differences caused by formatting.
# I stop execution if any other status appears.
assert pipelines_active_tx["STATUS_CD"].astype(
    str
).str.strip().eq("I").all()


# ---------------------------------------------------------
# I examine the completeness of the well inventory.
# ---------------------------------------------------------

# I print a heading for the RRC symbol-code distribution.
print("Well SYMNUM distribution:")

# I count the number of well records in each SYMNUM category.
# I include missing classifications to identify records
# that cannot be assigned to a known symbol category.
# I sort the results by SYMNUM to make them easier to review.
display(
    wells_tx["SYMNUM"]
    .value_counts(dropna=False)
    .sort_index()
)


# ---------------------------------------------------------
# I check for a potentially outdated well export.
# ---------------------------------------------------------

# I use one million records as a diagnostic threshold
# to flag a potentially outdated or restricted well inventory.
# This is a project-specific safeguard, not an official
# RRC record-count requirement.
# I do not assume that the total number of wells is permanent.
if len(wells_tx) < 1_000_000:

    # I stop execution if the well inventory falls below
    # the threshold because I may have loaded an older export.
    # I require further investigation before continuing
    # with the spatial analysis.
    raise ValueError(
        "I appear to have loaded the older filtered well inventory. "
        "I must rerun reviewed Notebook 01."
    )

Wind 19,943 EPSG:4326 {'Point': 19943} missing geometry 0
Solar 187 EPSG:4326 {'MultiPolygon': 152, 'Polygon': 35} missing geometry 0
Battery 200 EPSG:4326 {'Point': 200} missing geometry 0
Wells 1,380,352 EPSG:4326 {'Point': 1380352} missing geometry 0
Pipeline all 496,305 EPSG:4326 {'MultiLineString': 496305} missing geometry 0
Pipeline in service 435,979 EPSG:4326 {'MultiLineString': 435979} missing geometry 0
Well SYMNUM distribution:


SYMNUM
1           2
2       88272
3      288905
4      239950
5       79108
6       17410
7      325237
8       61541
9       39438
10      22481
11      13819
12        763
16          8
17         37
18         97
19       2884
20        827
21      24162
22        637
23        585
27          1
36          3
47          4
73         89
74        780
75        366
76         25
77         29
78        118
79        175
80         15
81        104
86     144958
87      26777
88        586
89         23
90          1
103        33
104         3
105         4
106         3
108         2
110         2
114         1
116        35
118         2
121         1
124         4
126         1
136        24
140         3
144         2
148         1
152        14
Name: count, dtype: int64

## County boundaries and point infrastructure

In [3]:
# I search only the known county-boundary locations used in the two earlier notebooks.
county_candidates = [RAW / "Texas_Boundaries" / "tl_2026_us_county",
                     PROJECT_ROOT / "Texas Boundries" / "tl_2026_us_county"]
county_source = next((p for p in county_candidates if p.exists()), None)
if county_source is None: raise FileNotFoundError(f"I cannot locate the county boundaries: {county_candidates}")
us_counties = gpd.read_file(county_source)
assert us_counties.crs is not None
texas_counties = us_counties.loc[us_counties["STATEFP"].astype(str).str.zfill(2).eq("48")].copy()
assert len(texas_counties) == 254 and texas_counties["GEOID"].is_unique
assert texas_counties.geometry.notna().all() and (~texas_counties.geometry.is_empty).all()
if not texas_counties.geometry.is_valid.all():
    raise ValueError("I need to review invalid Texas county boundaries before using them")
texas_counties_projected = texas_counties.to_crs("EPSG:3083")
county_lookup = texas_counties_projected[["GEOID","NAME","geometry"]].rename(
    columns={"GEOID":"county_geoid","NAME":"county_name"})
assert county_lookup.crs == texas_counties_projected.crs
print("I loaded",len(county_lookup),"Texas counties")

def assign_counties_to_points(frame, name):
    # I keep one output row per input point; points on boundaries need separate review.
    assert frame.crs is not None
    projected = frame.to_crs("EPSG:3083").drop(columns=["county_geoid","county_name","index_right"], errors="ignore")
    projected["source_row_id"] = np.arange(len(projected), dtype="int64")
    valid = projected.geometry.notna() & ~projected.geometry.is_empty
    if not projected.loc[valid].geom_type.eq("Point").all():
        raise ValueError(f"I expected point geometry for {name}")
    joined = gpd.sjoin(projected.loc[valid], county_lookup, how="left", predicate="within").drop(columns="index_right")
    if joined["source_row_id"].duplicated().any():
        raise ValueError(f"I found multiple county matches for one {name} record")
    # I preserve invalid or missing point geometry as unmatched rows.
    if (~valid).any():
        missing = projected.loc[~valid].copy()
        missing["county_geoid"] = None; missing["county_name"] = None
        joined = pd.concat([joined, missing], ignore_index=True)
    joined = gpd.GeoDataFrame(joined, geometry="geometry", crs=projected.crs).sort_values("source_row_id")
    assert len(joined) == len(frame)
    joined["county_match_status"] = np.where(joined["county_geoid"].notna(),"within county","unmatched or boundary")
    print(name, "matched", int(joined["county_geoid"].notna().sum()),"unmatched",int(joined["county_geoid"].isna().sum()))
    return joined

wind_counties = assign_counties_to_points(wind_tx,"wind turbines")
battery_counties = assign_counties_to_points(battery_tx,"battery plants")
wells_counties_final = assign_counties_to_points(wells_tx,"well surface records")
wells_counties_final["within_county_boundary"] = wells_counties_final["county_geoid"].notna()
# I do not treat county matching as evidence that a well is operating.
for name, frame in [("Wind",wind_counties),("Battery",battery_counties),("Wells",wells_counties_final)]:
    save_gpkg(frame, OUT / f"Texas_{name}_By_County.gpkg", name.lower())

I loaded 254 Texas counties
wind turbines matched 19943 unmatched 0
battery plants matched 200 unmatched 0
well surface records matched 1378797 unmatched 1555
I saved Texas_Wind_By_County.gpkg: 19,943 rows
I saved Texas_Battery_By_County.gpkg: 200 rows
I saved Texas_Wells_By_County.gpkg: 1,380,352 rows


In [4]:
# I check whether the TIGER/Line county dataset includes
# reported water area within Texas counties.

print("County boundary source:", county_source)
print("Number of Texas counties:", len(texas_counties))
print("Coordinate system:", texas_counties.crs)

# I inspect the available geographic attributes.
print("\nAvailable boundary columns:")
print(texas_counties.columns.tolist())

# I check the Census-reported land and water areas.
if {"ALAND", "AWATER"}.issubset(texas_counties.columns):

    county_water_check = texas_counties[
        ["NAME", "GEOID", "ALAND", "AWATER"]
    ].copy()

    county_water_check["water_sqmi"] = (
        pd.to_numeric(county_water_check["AWATER"])
        / 2_589_988.110336
    )

    county_water_check["land_sqmi"] = (
        pd.to_numeric(county_water_check["ALAND"])
        / 2_589_988.110336
    )

    county_water_check = county_water_check.sort_values(
        "water_sqmi", ascending=False
    )

    print("\nCounties reporting water area:",
          (county_water_check["water_sqmi"] > 0).sum())

    print("\nTotal Census-reported water area (sq mi):",
          round(county_water_check["water_sqmi"].sum(), 2))

    display(county_water_check.head(20))

else:
    print("ALAND and AWATER fields are unavailable.")

County boundary source: G:\My Drive\Projects\Large projects\2026TexasEnergyFootprint\01_Raw_Data\Texas_Boundaries\tl_2026_us_county
Number of Texas counties: 254
Coordinate system: EPSG:4269

Available boundary columns:
['STATEFP', 'COUNTYFP', 'COUNTYNS', 'GEOID', 'GEOIDFQ', 'NAME', 'NAMELSAD', 'LSAD', 'CLASSFP', 'MTFCC', 'CSAFP', 'CBSAFP', 'METDIVFP', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']

Counties reporting water area: 254

Total Census-reported water area (sq mi): 7339.55


,NAME,GEOID,ALAND,AWATER,water_sqmi,land_sqmi
381,Calhoun,48057,1312958746,1361633016,525.729447,506.936206
232,Matagorda,48321,2831196247,1345191837,519.381472,1093.130982
63,Galveston,48167,982674750,1280831816,494.531929,379.412842
3006,Kenedy,48261,3777657848,1261927316,487.232861,1458.561849
343,Cameron,48061,2309410447,1004962030,388.018009,891.668359
1534,Nueces,48355,2173963292,844859309,326.202003,839.371920
1535,Aransas,48007,653395811,714004934,275.678846,252.277533
62,Chambers,48071,1546824355,709405936,273.903163,597.232222
66,Brazoria,48039,3530976332,635433878,245.342392,1363.317584
233,Jefferson,48245,2270734693,611202804,235.986722,876.735566


In [10]:
## NEWNY2
# I compare the area of my county polygons with the
# Census-reported land and water areas.

# I use a projected coordinate system for area calculations.
counties_area_check = texas_counties.to_crs("EPSG:3083").copy()

# I calculate the area represented by the actual county geometries.
counties_area_check["geometry_sqmi"] = (
    counties_area_check.geometry.area / 2_589_988.110336
)

# I convert the Census land and water attributes to square miles.
counties_area_check["land_sqmi"] = (
    pd.to_numeric(counties_area_check["ALAND"])
    / 2_589_988.110336
)

counties_area_check["water_sqmi"] = (
    pd.to_numeric(counties_area_check["AWATER"])
    / 2_589_988.110336
)

counties_area_check["land_plus_water_sqmi"] = (
    counties_area_check["land_sqmi"]
    + counties_area_check["water_sqmi"]
)

# I compare the polygon area with both Census measures.
totals = counties_area_check[
    ["geometry_sqmi", "land_sqmi",
     "water_sqmi", "land_plus_water_sqmi"]
].sum()

print(totals.round(2))

print("\nDifference from land-only area (sq mi):",
      round(totals["geometry_sqmi"] - totals["land_sqmi"], 2))

print("\nDifference from land + water area (sq mi):",
      round(
          totals["geometry_sqmi"]
          - totals["land_plus_water_sqmi"], 2
      ))

geometry_sqmi           268599.06
land_sqmi               261259.49
water_sqmi                7339.55
land_plus_water_sqmi    268599.04
dtype: float64

Difference from land-only area (sq mi): 7339.57

Difference from land + water area (sq mi): 0.02


## Well classification

In [5]:
# I normalize the original RRC symbol codes without deleting any well records.
well_codes = pd.to_numeric(wells_counties_final["SYMNUM"], errors="coerce").astype("Int64")
wells_counties_final["symnum_normalized"] = well_codes

# I retain dry holes and all other categories in my authoritative spatial inventory.
dry_hole_mask = well_codes.eq(3).fillna(False)
print("Dry-hole records preserved in master inventory:", int(dry_hole_mask.sum()))
print("Complete well surface records:", len(wells_counties_final))
assert len(wells_counties_final) == len(wells_tx)
assert int(dry_hole_mask.sum()) == int(pd.to_numeric(wells_tx["SYMNUM"], errors="coerce").eq(3).sum())

# I document the historical exclusion codes without interpreting the remaining wells as producing.
legacy_exclusion_codes = {2, 3, 7, 8, 9, 10}
wells_counties_final["legacy_excluded_category"] = well_codes.isin(legacy_exclusion_codes).fillna(False)
wells_counties_final["well_scenario_label"] = np.where(
    wells_counties_final["legacy_excluded_category"],
    "Excluded from legacy restricted scenario",
    "Included in legacy restricted scenario (not verified operating)"
)

# I save the complete spatial file again with explicit scenario labels.
save_gpkg(wells_counties_final, OUT / "Texas_Wells_By_County.gpkg", "wells")

# I produce a separate restricted candidate file, never overwriting the master inventory.
wells_legacy_candidate = wells_counties_final.loc[
    ~wells_counties_final["legacy_excluded_category"]
].copy()
save_gpkg(wells_legacy_candidate, OUT / "Texas_Wells_Legacy_Restricted_Candidate.gpkg", "wells_candidate")

# I retain a full count audit, including unmatched county locations and missing classifications.
well_category_audit = (
    wells_counties_final.groupby(
        ["symnum_normalized", "well_category", "county_match_status", "legacy_excluded_category"],
        dropna=False, observed=True
    ).size().rename("record_count").reset_index()
)
well_category_audit.to_csv(OUT / "Texas_Wells_Category_County_Audit.csv", index=False)
assert int(well_category_audit["record_count"].sum()) == len(wells_counties_final)
assert len(wells_legacy_candidate) + int(wells_counties_final["legacy_excluded_category"].sum()) == len(wells_counties_final)
print("Legacy restricted candidate records:", len(wells_legacy_candidate))
print("Legacy excluded-category records (still retained in master):", int(wells_counties_final["legacy_excluded_category"].sum()))
display(well_category_audit.groupby(["symnum_normalized", "well_category"], dropna=False)["record_count"].sum().reset_index())

Dry-hole records preserved in master inventory: 288905
Complete well surface records: 1380352
I saved Texas_Wells_By_County.gpkg: 1,380,352 rows
I saved Texas_Wells_Legacy_Restricted_Candidate.gpkg: 554,478 rows
Legacy restricted candidate records: 554478
Legacy excluded-category records (still retained in master): 825874


,symnum_normalized,well_category,record_count
0,1,Unknown,2
1,2,Permitted location,88272
2,3,Dry hole,288905
3,4,Oil/gas well,239950
4,5,Oil/gas well,79108
5,6,Oil/gas well,17410
6,7,Plugged well,325237
7,8,Plugged well,61541
8,9,Canceled location,39438
9,10,Plugged well,22481


In [6]:

# I investigate why some well surface records
# were not assigned to a Texas county.

unmatched_wells = wells_counties_final[
    wells_counties_final["county_geoid"].isna()
].copy()

print("Total unmatched well records:", len(unmatched_wells))

# I distinguish missing geometries from valid points
# that did not fall strictly within a county polygon.
missing_geometry = (
    unmatched_wells.geometry.isna()
    | unmatched_wells.geometry.is_empty
)

print("Missing or empty geometries:", missing_geometry.sum())

valid_unmatched = unmatched_wells.loc[~missing_geometry].copy()

# I check whether unmatched points intersect a county boundary.
# This helps identify points that 'within' may have missed.
boundary_matches = gpd.sjoin(
    valid_unmatched,
    county_lookup,
    how="inner",
    predicate="intersects",
    lsuffix="well",
    rsuffix="county"
)

print(
    "Unique unmatched wells intersecting a county:",
    boundary_matches["source_row_id"].nunique()
)

# I summarize the original well classification codes
# to understand which types of wells are affected.
print("\nUnmatched well classifications:")

display(
    unmatched_wells["SYMNUM"]
    .value_counts(dropna=False)
    .head(20)
)

Total unmatched well records: 1555
Missing or empty geometries: 0
Unique unmatched wells intersecting a county: 0

Unmatched well classifications:


SYMNUM
3     534
87    488
2     159
8     155
9     141
5      35
7      13
10     11
6      10
4       4
86      3
21      1
20      1
Name: count, dtype: int64

## Solar polygons: county intersections and unique footprint

In [7]:

# ---------------------------------------------------------
# 1. I validate the solar geometries and calculate their area.
# ---------------------------------------------------------

# I transform the solar project geometries to EPSG:3083.
# I use this projected CRS to calculate area in square meters.
# I create a copy to preserve the original solar_tx dataset.
solar_projected = solar_tx.to_crs("EPSG:3083").copy()

# I check whether each geometry exists and is not empty.
# .notna() identifies nonmissing geometries.
# .is_empty identifies geometries containing no spatial features.
# The ~ operator reverses the Boolean values.
# The & operator requires both conditions to be True.
valid = (
    solar_projected.geometry.notna()
    & ~solar_projected.geometry.is_empty
)

# I check whether every geometry is present, nonempty, and valid.
# .all() returns True only if every record passes the check.
# The not operator reverses that result.
# The or operator triggers the error if either validation fails.
if not valid.all() or not solar_projected.geometry.is_valid.all():

    # I stop execution if invalid or missing geometries remain.
    # I expect geometry repairs to have been completed in Notebook 01.
    raise ValueError(
        "I need to repair or investigate invalid solar polygons in Notebook 01"
    )

# I verify that every solar geometry is a Polygon or MultiPolygon.
# .geom_type retrieves each geometry's type.
# .isin() checks whether each type belongs to the allowed list.
assert solar_projected.geom_type.isin(
    ["Polygon", "MultiPolygon"]
).all()

# I assign a sequential identifier to every original solar record.
# np.arange() generates integers starting at zero.
# I use these identifiers to track projects after county clipping.
solar_projected["solar_record_id"] = np.arange(
    len(solar_projected),
    dtype="int64"
)

# I calculate the area of each solar polygon in square meters.
# I multiply by ACRES_PER_SQM to convert the result to acres.
solar_projected["polygon_acres"] = (
    solar_projected.geometry.area * ACRES_PER_SQM
)


# ---------------------------------------------------------
# 2. I intersect solar projects with Texas counties.
# ---------------------------------------------------------

# I calculate the intersection between solar polygons
# and county polygons using GeoPandas.
# how="intersection" retains only overlapping areas.
# A solar project crossing a county boundary can be split
# into multiple records, each associated with a county.
# keep_geom_type=True retains polygon-type intersections.
solar_counties = gpd.overlay(
    solar_projected,
    county_lookup,
    how="intersection",
    keep_geom_type=True
)

# I calculate the area of each county-clipped solar polygon.
# I convert square meters to acres.
solar_counties["county_solar_acres"] = (
    solar_counties.geometry.area * ACRES_PER_SQM
)

# I retain only county-clipped polygons with positive area.
# .loc[] selects rows satisfying the Boolean condition.
# .copy() creates an independent GeoDataFrame.
solar_counties = solar_counties.loc[
    solar_counties["county_solar_acres"] > 0
].copy()

# I print the number of original solar records.
# I also count how many distinct original records remain
# represented after the county intersection.
# .nunique() counts unique solar_record_id values,
# rather than counting all county-clipped pieces.
print(
    "Original projects",
    len(solar_projected),
    "represented after clipping",
    solar_counties["solar_record_id"].nunique()
)

# I compare the summed area of the original solar polygons
# with the summed area of the county-clipped polygons.
# This helps me identify area that was not assigned to counties.
print(
    "Summed original acres",
    solar_projected["polygon_acres"].sum(),
    "clipped project acres",
    solar_counties["county_solar_acres"].sum()
)


# ---------------------------------------------------------
# 3. I calculate unique solar polygon coverage by county.
# ---------------------------------------------------------

# I create an empty list to collect each county's unioned geometry.
unique_rows = []

# I iterate through the county polygons one at a time.
# .itertuples(index=False) provides each row as a named tuple.
# tqdm() displays the progress of the county calculations.
# total tells tqdm how many county records to expect.
for row in tqdm(
    county_lookup.itertuples(index=False),
    total=len(county_lookup),
    desc="Unioning solar polygons by county"
):

    # I select the solar polygon pieces assigned to this county.
    # .eq() compares each county_geoid with the current county ID.
    # .loc[] selects the matching rows.
    # I retrieve only their geometry column.
    pieces = solar_counties.loc[
        solar_counties["county_geoid"].eq(row.county_geoid),
        "geometry"
    ]

    # I check whether the current county contains solar polygons.
    if len(pieces):

        # I convert the selected geometries to a Python list.
        # unary_union() combines them into one geometric result.
        # Overlapping polygon areas are represented only once
        # in the resulting union.
        geom = unary_union(pieces.tolist())

        # I add the county ID, county name, unioned area,
        # and resulting geometry to the output list.
        # I calculate the unioned area in acres.
        unique_rows.append({
            "county_geoid": row.county_geoid,
            "county_name": row.county_name,
            "unique_solar_acres": geom.area * ACRES_PER_SQM,
            "geometry": geom
        })

# I convert the collected county records into a GeoDataFrame.
# I identify geometry as the spatial column.
# I retain EPSG:3083 as the coordinate reference system.
solar_unique_counties = gpd.GeoDataFrame(
    unique_rows,
    geometry="geometry",
    crs="EPSG:3083"
)

# I verify that the total unioned area does not exceed
# the summed area of the county-clipped solar polygons.
# I allow a tolerance of 0.01 acres for numerical differences.
# I expect the unioned area to be equal to or smaller
# because overlapping areas are counted only once.
assert (
    solar_unique_counties["unique_solar_acres"].sum()
    <= solar_counties["county_solar_acres"].sum() + 0.01
)


# ---------------------------------------------------------
# 4. I summarize solar projects and acreage by county.
# ---------------------------------------------------------

# I group the county-clipped solar records by county ID and name.
# as_index=False keeps those identifiers as regular columns.
solar_county_summary = solar_counties.groupby(
    ["county_geoid", "county_name"],
    as_index=False
).agg(

    # I count all county-clipped polygon records.
    # This measures polygon pieces, not necessarily unique projects.
    project_parts=("solar_record_id", "size"),

    # I count distinct original solar record identifiers per county.
    # This avoids counting multiple pieces of the same source record
    # as separate projects within that county.
    distinct_projects=("solar_record_id", "nunique"),

    # I sum the area of all county-clipped solar polygons.
    # This total may count overlapping project areas more than once.
    summed_project_acres=("county_solar_acres", "sum")
)

# I merge the county summary with the unique footprint results.
# I remove the geometry column because this output is a table.
# I match records using county_geoid and county_name.
# validate="one_to_one" raises an error if either table
# contains duplicate county identifiers for the merge.
solar_county_summary = solar_county_summary.merge(
    solar_unique_counties.drop(columns="geometry"),
    on=["county_geoid", "county_name"],
    validate="one_to_one"
)

# I calculate the difference between summed polygon acreage
# and unique unioned acreage for each county.
# This measures the area counted more than once
# when overlapping source polygons are summed.
solar_county_summary["overlap_acres"] = (
    solar_county_summary["summed_project_acres"]
    - solar_county_summary["unique_solar_acres"]
)


# ---------------------------------------------------------
# 5. I save the county-level solar outputs.
# ---------------------------------------------------------

# I save the individual solar polygon pieces assigned to counties.
# I preserve their original record identifiers and county attributes.
save_gpkg(
    solar_counties,
    OUT / "Texas_Solar_By_County.gpkg",
    "solar_parts"
)

# I save the unioned solar geometry for each county.
# This represents unique mapped polygon coverage.
save_gpkg(
    solar_unique_counties,
    OUT / "Texas_Solar_Unique_Footprint_By_County.gpkg",
    "solar_unique"
)

# I save the county-level summary as a CSV file.
# index=False prevents pandas from exporting its row index.
solar_county_summary.to_csv(
    OUT / "Texas_Solar_County_Summary.csv",
    index=False
)


Original projects 187 represented after clipping 187
Summed original acres 143025.17307269847 clipped project acres 143025.17307269835


Unioning solar polygons by county:   0%|          | 0/254 [00:00<?, ?it/s]

I saved Texas_Solar_By_County.gpkg: 197 rows
I saved Texas_Solar_Unique_Footprint_By_County.gpkg: 86 rows


## Pipelines: county-clipped mileage

In [8]:

# ---------------------------------------------------------
# 1. I prepare the original pipeline geometries.
# ---------------------------------------------------------

# I transform all pipeline geometries to EPSG:3083.
# I use this projected CRS because its coordinate units are meters.
# I create a copy so I do not modify pipelines_all_tx.
pipelines_all_projected = pipelines_all_tx.to_crs("EPSG:3083").copy()

# I create a new column containing sequential integers from 0
# to the number of pipeline records minus 1.
# np.arange() generates the numbers, and dtype="int64"
# stores them as 64-bit integers.
# I use these IDs to reconnect county-clipped pieces
# to their original pipeline records.
pipelines_all_projected["pipeline_record_id"] = np.arange(
    len(pipelines_all_projected),
    dtype="int64"
)

# I check that no pipeline geometry is missing.
# .notna() returns True for nonmissing geometries.
# .all() requires every record to satisfy that condition.
# I also check that no geometry is empty.
# The ~ operator reverses the True/False values returned
# by .is_empty, so True means the geometry is not empty.
# The assert statement stops execution if either check fails.
assert (
    pipelines_all_projected.geometry.notna().all()
    and (~pipelines_all_projected.geometry.is_empty).all()
)

# I check that every geometry is either a LineString
# or a MultiLineString.
# .isin() checks each geometry type against the allowed list.
# .all() confirms that every record has an allowed type.
assert pipelines_all_projected.geom_type.isin(
    ["LineString", "MultiLineString"]
).all()

# I check that every pipeline geometry satisfies
# Shapely's geometry-validity rules.
# I stop execution if any geometry is invalid.
assert pipelines_all_projected.geometry.is_valid.all()

# I calculate the length of each pipeline geometry.
# .geometry.length returns the length in CRS units (meters).
# I divide by M_PER_MILE (1609.344) to convert meters to miles.
# I save the results in a new column.
pipelines_all_projected["mapped_miles_recalculated"] = (
    pipelines_all_projected.geometry.length / M_PER_MILE
)

# I check whether the dataset contains a column named geometry_miles.
# This condition prevents an error if the column is absent.
if "geometry_miles" in pipelines_all_projected:

    # I convert the stored length values to numeric values.
    # errors="coerce" converts values that cannot be interpreted
    # as numbers to NaN rather than raising an error.
    # .sum() adds the numeric values, excluding NaN by default.
    # I compare the stored total with my recalculated total.
    print(
        "Stored vs recalculated mapped miles",
        pd.to_numeric(
            pipelines_all_projected["geometry_miles"],
            errors="coerce"
        ).sum(),
        pipelines_all_projected["mapped_miles_recalculated"].sum()
    )

# I calculate and print total mapped mileage across all statuses.
print(
    "All status miles",
    pipelines_all_projected["mapped_miles_recalculated"].sum()
)

# I count records with STATUS_CD equal to "I".
# .astype(str) converts the status values to strings.
# .str.strip() removes spaces at the beginning and end.
# .eq("I") returns True for records classified as in service.
# .sum() counts the True values because True equals 1 in Python.
print(
    "In-service records",
    pipelines_all_projected["STATUS_CD"]
    .astype(str)
    .str.strip()
    .eq("I")
    .sum()
)


# ---------------------------------------------------------
# 2. I define a function to detect changes to input files.
# ---------------------------------------------------------

# I define a reusable function that calculates
# a SHA-256 fingerprint for a file.
def file_fingerprint(path):

    # I initialize a SHA-256 hashing object.
    h = hashlib.sha256()

    # I open the file in binary-read mode ("rb").
    # I use with so Python automatically closes the file.
    with open(path, "rb") as f:

        # I read the file in chunks of 8 megabytes.
        # The lambda function reads the next chunk.
        # iter() repeatedly calls the function until it returns b"",
        # which indicates that the end of the file has been reached.
        # This avoids loading the entire file into memory.
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b""
        ):

            # I add the bytes from each chunk to the hash calculation.
            h.update(chunk)

    # I return the final SHA-256 fingerprint
    # as a hexadecimal string.
    return h.hexdigest()


# ---------------------------------------------------------
# 3. I define the cache and input file locations.
# ---------------------------------------------------------

# I define the GeoPackage file that stores the
# previously calculated pipeline-county intersections.
pipeline_cache = OUT / "Texas_Pipelines_By_County.gpkg"

# I define the JSON file containing the cache metadata.
cache_meta = OUT / "Texas_Pipelines_By_County.cache.json"

# I identify the cleaned pipeline input from Notebook 01.
source_pipeline = CLEAN / "Texas_Pipelines_All_Statuses.gpkg"

# I determine which county source file to use for fingerprinting.
# If county_source is a file, I use it directly.
# Otherwise, I search that directory for .shp files
# and select the first one returned by glob().
source_county = (
    county_source
    if county_source.is_file()
    else next(county_source.glob("*.shp"))
)


# ---------------------------------------------------------
# 4. I create a signature describing the current inputs.
# ---------------------------------------------------------

# I create a dictionary containing the fingerprints
# and processing information used to validate the cache.
cache_signature = {

    # I calculate a SHA-256 fingerprint of the pipeline file.
    "pipeline_sha256": file_fingerprint(source_pipeline),

    # I calculate a SHA-256 fingerprint of the county .shp file.
    "county_shp_sha256": file_fingerprint(source_county),

    # I record the number of county features in county_lookup.
    "county_geoid_count": len(county_lookup),

    # I record a manually assigned processing version.
    # I can change this string when the intersection logic changes
    # to prevent reuse of results from an older implementation.
    "processing_version": "county-intersection-v3-2026-11"
}


# ---------------------------------------------------------
# 5. I decide whether the existing cache can be reused.
# ---------------------------------------------------------

# I evaluate four conditions using the and operator.
# Python returns True only if all four conditions are True.
cache_ok = (

    # I check that I have not requested a forced rebuild.
    not REBUILD_PIPELINE_CLIP

    # I check that the cached GeoPackage exists.
    and pipeline_cache.exists()

    # I check that the cache metadata JSON exists.
    and cache_meta.exists()

    # I read the JSON file as text and convert it to a dictionary.
    # I compare that dictionary with the current cache signature.
    # I reuse the cache only if the dictionaries are equal.
    and json.loads(cache_meta.read_text()) == cache_signature
)

# I perform additional checks if the initial cache test passes.
if cache_ok:

    # I load the existing county-clipped pipeline layer.
    pipelines_counties = gpd.read_file(
        pipeline_cache,
        layer="pipeline_parts"
    )

    # I define the columns required for later analysis.
    needed = {
        "pipeline_record_id",
        "county_geoid",
        "county_pipeline_miles",
        "STATUS_CD",
        "SYSTYPE"
    }

    # I verify that all required columns exist.
    # I also require the cached dataset to contain at least one row.
    # I update cache_ok with the result of these checks.
    cache_ok = (
        needed.issubset(pipelines_counties.columns)
        and len(pipelines_counties) > 0
    )


# ---------------------------------------------------------
# 6. I reuse the cache or rebuild the county intersections.
# ---------------------------------------------------------

# I check whether the cached output passed all validation conditions.
if cache_ok:

    # I reuse the loaded pipeline-county intersections.
    # I print the number of county-clipped records.
    print(
        "I reused the validated pipeline clip",
        len(pipelines_counties)
    )

else:

    # I divide the pipeline dataset into batches
    # to limit intermediate memory use.
    # Each batch contains at most 10,000 original records.
    batch_size = 10000

    # I create an empty list to collect the clipped results.
    parts = []

    # I generate starting row positions:
    # 0, 10000, 20000, and so on.
    # range() stops before reaching the total record count.
    # tqdm() displays progress as the batches are processed.
    for start in tqdm(
        range(0, len(pipelines_all_projected), batch_size),
        desc="Clipping pipelines to counties"
    ):

        # I select up to 10,000 records using integer-position slicing.
        # .iloc includes the starting position but excludes the end.
        batch = pipelines_all_projected.iloc[
            start:start + batch_size
        ]

        # I calculate the geometric intersection between
        # the pipeline batch and the county polygons.
        # how="intersection" retains only overlapping geometries.
        # A pipeline crossing several counties can produce
        # separate line pieces, each with county attributes.
        # keep_geom_type=True retains line-type results
        # and discards intersections of different geometry types.
        clipped = gpd.overlay(
            batch,
            county_lookup,
            how="intersection",
            keep_geom_type=True
        )

        # I check whether the current batch produced any intersections.
        if len(clipped):

            # I calculate the length of each county-clipped piece
            # and convert it from meters to miles.
            clipped["county_pipeline_miles"] = (
                clipped.geometry.length / M_PER_MILE
            )

            # I retain only pieces with positive mapped length.
            # This removes zero-length results from the allocation.
            clipped = clipped.loc[
                clipped["county_pipeline_miles"] > 0
            ]

            # I append the current batch's results to my list.
            parts.append(clipped)

    # I stop execution if no batch produced any intersections.
    # This could indicate a problem with the CRS, geometries,
    # or geographic coverage of the input datasets.
    if not parts:
        raise ValueError(
            "I found no pipeline intersections; "
            "I need to inspect CRS and boundaries"
        )

    # I combine the clipped batches into one table.
    # pd.concat() joins the rows vertically.
    # ignore_index=True creates a new sequential index.
    # I construct a GeoDataFrame using the geometry column
    # and assign EPSG:3083 as its CRS.
    pipelines_counties = gpd.GeoDataFrame(
        pd.concat(parts, ignore_index=True),
        geometry="geometry",
        crs="EPSG:3083"
    )

    # I save the newly calculated county intersections
    # so I can reuse them in later notebook runs.
    save_gpkg(
        pipelines_counties,
        pipeline_cache,
        "pipeline_parts"
    )

    # I convert the cache signature dictionary to JSON text.
    # indent=2 makes the metadata easier to read.
    # I save the metadata alongside the GeoPackage.
    cache_meta.write_text(
        json.dumps(cache_signature, indent=2)
    )


# ---------------------------------------------------------
# 7. I separate in-service pipeline segments.
# ---------------------------------------------------------

# I select county-clipped records whose RRC status code is "I".
# .loc filters rows using the Boolean condition.
# .copy() creates a separate GeoDataFrame.
# I preserve all statuses in pipelines_counties.
pipelines_active_counties = pipelines_counties.loc[
    pipelines_counties["STATUS_CD"]
    .astype(str)
    .str.strip()
    .eq("I")
].copy()


# ---------------------------------------------------------
# 8. I compare original and county-allocated mileage.
# ---------------------------------------------------------

# I sum the recalculated lengths of all original pipeline records.
original_miles = pipelines_all_projected[
    "mapped_miles_recalculated"
].sum()

# I sum the lengths of all county-clipped pipeline pieces.
county_miles = pipelines_counties[
    "county_pipeline_miles"
].sum()

# I print both totals to examine the overall allocation.
print(
    "Original segment miles",
    original_miles,
    "county-clipped segment miles",
    county_miles
)

# I check whether the county-allocated mileage exceeds
# the original mileage by more than 0.1%.
# Multiplying by 1.001 allows that relative tolerance.
if county_miles > original_miles * 1.001:

    # I stop execution because excess mileage could indicate
    # overlapping county polygons or double counting.
    raise ValueError(
        "I found potential county-boundary double counting "
        "or overlapping county geometries"
    )


# ---------------------------------------------------------
# 9. I check over-allocation for each original record.
# ---------------------------------------------------------

# I group county-clipped pieces by their original pipeline ID.
# I sum the mileage assigned to counties for each source record.
per_source = pipelines_counties.groupby(
    "pipeline_record_id"
)["county_pipeline_miles"].sum()

# I create a Series containing the original length
# of each pipeline record, indexed by pipeline_record_id.
# set_index() allows me to look up lengths by source ID.
original = pipelines_all_projected.set_index(
    "pipeline_record_id"
)["mapped_miles_recalculated"]

# I align original lengths with the IDs in per_source.
# reindex() rearranges the original Series to match those IDs.
# I subtract original mileage from county-allocated mileage.
# A positive difference means the allocated mileage is greater.
# .loc[lambda s: ...] retains differences above 0.001 miles.
excess = (
    per_source - original.reindex(per_source.index)
).loc[
    lambda s: s > 0.001
]

# I stop execution if any source record exceeds
# its original mileage by more than 0.001 miles.
if len(excess):
    raise ValueError(
        f"I found {len(excess)} pipeline records "
        "counted more than once across counties"
    )


# ---------------------------------------------------------
# 10. I save the in-service county pipeline dataset.
# ---------------------------------------------------------

# I export only county-clipped pipeline pieces classified
# as in service under the RRC status field.
save_gpkg(
    pipelines_active_counties,
    OUT / "Texas_Pipelines_In_Service_By_County.gpkg",
    "pipeline_in_service"
)


# ---------------------------------------------------------
# 11. I summarize pipeline mileage by county and status.
# ---------------------------------------------------------

# I group the county-clipped dataset by county GEOID,
# county name, and RRC status code.
# dropna=False retains groups with missing classification values.
# as_index=False keeps the grouping fields as regular columns.
pipeline_county_summary = pipelines_counties.groupby(
    ["county_geoid", "county_name", "STATUS_CD"],
    dropna=False,
    as_index=False
).agg(

    # I count the number of county-clipped pieces in each group.
    # This counts segment parts, not necessarily unique pipelines.
    segment_parts=("pipeline_record_id", "size"),

    # I sum the mapped mileage of those pieces.
    mapped_miles=("county_pipeline_miles", "sum")
)

# I save the county-by-status summary as a CSV file.
# index=False prevents pandas from adding an index column.
pipeline_county_summary.to_csv(
    OUT / "Texas_Pipelines_County_Status_Summary.csv",
    index=False
)

Stored vs recalculated mapped miles 363844.1082118045 363844.10821180267
All status miles 363844.10821180267
In-service records 435979
I reused the validated pipeline clip 507807
Original segment miles 363844.10821180267 county-clipped segment miles 362845.15606194345
I saved Texas_Pipelines_In_Service_By_County.gpkg: 446,257 rows


##  outputs

In [9]:
# I confirm that county joins preserve source point counts and that the analysis outputs exist.
checks={"wind_source":len(wind_tx),"wind_county":len(wind_counties),
        "battery_source":len(battery_tx),"battery_county":len(battery_counties),
        "well_source":len(wells_tx),"well_county":len(wells_counties_final),
        "well_unmatched":int(wells_counties_final["county_geoid"].isna().sum()),
        "dry_holes_retained":int(dry_hole_mask.sum()),
        "legacy_restricted_well_candidate":len(wells_legacy_candidate),
        "solar_projects":len(solar_tx),"solar_county_parts":len(solar_counties),
        "solar_unique_county_acres":float(solar_unique_counties["unique_solar_acres"].sum()),
        "pipeline_all_county_parts":len(pipelines_counties),
        "pipeline_in_service_county_parts":len(pipelines_active_counties),
        "fossil_full_inventory":len(fossil_inventory),"fossil_map_ready":len(fossil_plants_tx),
        "refinery_inventory":len(refineries),"refinery_location_candidates":len(refineries_counties),
        "uic_administrative_records":len(disposal_wells)}
assert len(wind_counties)==len(wind_tx)
assert len(battery_counties)==len(battery_tx)
assert len(wells_counties_final)==len(wells_tx)
assert len(fossil_plants_counties)==len(fossil_plants_tx)
pd.Series(checks).rename("value").to_csv(OUT/"Notebook02_Reconciliation.csv")
display(pd.Series(checks).rename("value"))

NameError: name 'fossil_inventory' is not defined

In [ ]:
# ---------------------------------------------------------
# 1. I count county-clipped pieces for each pipeline record.
# ---------------------------------------------------------

# I group the county-clipped pipeline dataset by the original
# pipeline_record_id, which was assigned before county clipping.
# .groupby() places rows with the same identifier in one group.
# .size() counts how many rows belong to each group.
# The result is a Series indexed by pipeline_record_id.
# Records with no county intersection do not appear in this Series.
allocation_counts = (
    pipelines_counties
    .groupby("pipeline_record_id")
    .size()
)


# ---------------------------------------------------------
# 2. I add the county intersection count to the audit table.
# ---------------------------------------------------------

# I create a new column containing the number of county-clipped
# pieces associated with each original pipeline record.
pipeline_audit["county_clip_count"] = (

    # I select the original pipeline identifiers from the audit table.
    pipeline_audit["pipeline_record_id"]

    # I use .map() to look up each identifier in allocation_counts.
    # If an identifier is absent, .map() returns NaN.
    .map(allocation_counts)

    # I replace missing counts with zero.
    # A missing count means no matching county-clipped row was found.
    .fillna(0)

    # I convert the counts to integers because they represent
    # numbers of records rather than continuous measurements.
    .astype(int)
)


# ---------------------------------------------------------
# 3. I correct the pipeline allocation classifications.
# ---------------------------------------------------------

# I create a new classification column using np.select().
# np.select() evaluates the conditions in the order listed.
# It assigns the label corresponding to the first True condition.
# If no condition is True, it assigns the default label.
pipeline_audit["allocation_status_corrected"] = np.select(
    [
        # I identify original pipeline records that produced
        # no county-clipped rows.
        # .eq(0) returns True when county_clip_count equals zero.
        pipeline_audit["county_clip_count"].eq(0),

        # I identify records whose allocated mileage exceeds
        # their original mileage by more than 0.001 miles.
        # Because unallocated_miles = original - allocated,
        # a negative value indicates over-allocation.
        pipeline_audit["unallocated_miles"] < -0.001,

        # I identify records whose original mileage exceeds
        # county-allocated mileage by more than 0.001 miles.
        # These records have some mileage not assigned to counties.
        pipeline_audit["unallocated_miles"] > 0.001
    ],
    [
        # I assign this label when no county-clipped rows exist.
        "No county-clipped records",

        # I assign this label when county allocation exceeds
        # original mapped mileage beyond the tolerance.
        "Overallocated",

        # I assign this label when some original mileage
        # remains unallocated beyond the tolerance.
        "Partially allocated"
    ],

    # I classify remaining records as fully allocated
    # within the selected numerical tolerance.
    default="Fully allocated within tolerance"
)


# ---------------------------------------------------------
# 4. I summarize the corrected allocation classifications.
# ---------------------------------------------------------

# I group the source-record audit by the corrected status.
# dropna=False retains missing status values if any exist.
corrected_summary = (
    pipeline_audit
    .groupby("allocation_status_corrected", dropna=False)
    .agg(

        # I count original pipeline records in each category.
        source_records=("pipeline_record_id", "size"),

        # I sum the original mapped mileage in each category.
        original_miles=("mapped_miles_recalculated", "sum"),

        # I sum the mileage assigned to counties in each category.
        allocated_miles=("allocated_miles", "sum"),

        # I sum the differences between original and allocated
        # mileage in each category.
        unallocated_miles=("unallocated_miles", "sum")
    )

    # I convert the grouping index back into a regular column.
    .reset_index()
)


# ---------------------------------------------------------
# 5. I display the corrected summary.
# ---------------------------------------------------------

# I print a heading to identify the diagnostic output.
print("CORRECTED PIPELINE ALLOCATION SUMMARY")

# I display the summary as a formatted Jupyter table.
display(corrected_summary)


# ---------------------------------------------------------
# 6. I save the corrected diagnostic outputs.
# ---------------------------------------------------------

# I overwrite the existing source-record audit CSV.
# The updated file includes county_clip_count and
# allocation_status_corrected alongside the earlier fields.
# index=False prevents pandas from exporting its row index.
pipeline_audit.to_csv(
    diagnostic_dir / "Pipeline_Source_Record_Allocation_Audit.csv",
    index=False
)

# I overwrite the previous allocation summary CSV
# with the corrected classification results.
corrected_summary.to_csv(
    diagnostic_dir / "Pipeline_Final_Allocation_Summary.csv",
    index=False
)

In [ ]:
# I import Path to work with project file and folder paths.
from pathlib import Path

# I import NumPy for numerical comparisons and conditional classifications.
import numpy as np

# I import pandas to summarize, audit, and export tabular data.
import pandas as pd

# I import display to inspect the audit tables in Jupyter.
from IPython.display import display


# ---------------------------------------------------------
# I create a dedicated folder for diagnostic outputs.
# ---------------------------------------------------------

# I keep validation files separate from the main spatial outputs.
diagnostic_dir = OUT / "Diagnostics"

# I create the folder if it does not already exist.
diagnostic_dir.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------
# I verify that the required pipeline fields are available.
# ---------------------------------------------------------

# I list the fields needed from the original pipeline inventory.
# I need the record identifier, original mapped mileage,
# operating status, and pipeline system type.
required_original = {
    "pipeline_record_id",
    "mapped_miles_recalculated",
    "STATUS_CD",
    "SYSTYPE"
}

# I list the fields needed from the county-clipped pipeline dataset.
# I use these fields to reconnect each county segment
# to its original pipeline record.
required_clipped = {
    "pipeline_record_id",
    "county_pipeline_miles"
}

# I stop execution if any required original field is missing.
assert required_original.issubset(pipelines_all_projected.columns)

# I stop execution if any required county-clipped field is missing.
assert required_clipped.issubset(pipelines_counties.columns)

# I verify that each original pipeline record has a unique identifier.
# I need this to avoid confusing separate source records.
assert pipelines_all_projected["pipeline_record_id"].is_unique


# ---------------------------------------------------------
# I calculate county-allocated mileage by source record.
# ---------------------------------------------------------

# I group the county-clipped pieces by their original record ID.
# I sum their lengths because one pipeline can cross multiple counties.
allocated = (
    pipelines_counties
    .groupby("pipeline_record_id")["county_pipeline_miles"]
    .sum()
)

# I create an audit table from the complete original inventory.
# I preserve records even when they have no county intersection.
pipeline_audit = pipelines_all_projected[
    [
        "pipeline_record_id",
        "STATUS_CD",
        "SYSTYPE",
        "mapped_miles_recalculated"
    ]
].copy()

# I match each original record with its total county-allocated mileage.
# I assign zero when a record has no county-clipped segments.
pipeline_audit["allocated_miles"] = (
    pipeline_audit["pipeline_record_id"]
    .map(allocated)
    .fillna(0.0)
)

# I subtract allocated mileage from the original mapped mileage.
# A positive difference indicates mileage not assigned to counties.
# A negative difference indicates possible over-allocation.
pipeline_audit["unallocated_miles"] = (
    pipeline_audit["mapped_miles_recalculated"]
    - pipeline_audit["allocated_miles"]
)


# ---------------------------------------------------------
# I classify pipeline records by their allocation results.
# ---------------------------------------------------------

# I use a tolerance of 0.001 miles, approximately 1.61 meters.
# I allow this small difference to avoid treating minor
# numerical differences as meaningful discrepancies.
tolerance = 0.001

# I classify each original pipeline record.
# I distinguish records with almost no county allocation,
# partial allocation, excessive allocation, and full allocation.
pipeline_audit["allocation_status"] = np.select(
    [
        # I identify records with no meaningful allocated mileage.
        pipeline_audit["allocated_miles"] <= tolerance,

        # I identify records with original mileage remaining unallocated.
        pipeline_audit["unallocated_miles"] > tolerance,

        # I identify records whose county mileage exceeds original mileage.
        pipeline_audit["unallocated_miles"] < -tolerance
    ],
    [
        "Entirely outside county allocation",
        "Partially outside county allocation",
        "Overallocated to counties"
    ],
    default="Fully allocated"
)


# ---------------------------------------------------------
# I summarize the geographic allocation results.
# ---------------------------------------------------------

# I group records by allocation category.
# I calculate source-record counts and mileage totals
# to understand how much pipeline length falls into each category.
allocation_summary = (
    pipeline_audit
    .groupby("allocation_status", dropna=False)
    .agg(
        source_records=("pipeline_record_id", "size"),
        original_miles=("mapped_miles_recalculated", "sum"),
        allocated_miles=("allocated_miles", "sum"),
        unallocated_miles=("unallocated_miles", "sum")
    )
    .reset_index()
)

# I display the allocation summary for review.
print("PIPELINE ALLOCATION SUMMARY")
display(allocation_summary)


# ---------------------------------------------------------
# I reconcile statewide pipeline mileage.
# ---------------------------------------------------------

# I calculate the total original mapped pipeline mileage.
original_total = pipeline_audit["mapped_miles_recalculated"].sum()

# I calculate the total mileage assigned to counties.
allocated_total = pipeline_audit["allocated_miles"].sum()

# I calculate the net difference between original and allocated mileage.
unallocated_total = pipeline_audit["unallocated_miles"].sum()

# I display the statewide totals to assess the allocation results.
print("\nMILEAGE RECONCILIATION")
print(f"Original mapped mileage: {original_total:,.2f}")
print(f"County-allocated mileage: {allocated_total:,.2f}")
print(f"Unallocated mileage: {unallocated_total:,.2f}")


# ---------------------------------------------------------
# I examine records without meaningful county allocation.
# ---------------------------------------------------------

# I isolate records classified as entirely outside allocation.
# I use this to investigate the earlier finding of 341 records.
# I do not assume that the historical count must remain unchanged.
entirely_outside = pipeline_audit.loc[
    pipeline_audit["allocation_status"]
    == "Entirely outside county allocation"
]

# I report the number and original mileage of these records.
print("\nENTIRELY OUTSIDE COUNTY ALLOCATION")
print(f"Source records: {len(entirely_outside):,}")
print(
    "Original mileage:",
    f"{entirely_outside['mapped_miles_recalculated'].sum():,.2f}"
)


# ---------------------------------------------------------
# I examine records with partially unallocated mileage.
# ---------------------------------------------------------

# I identify pipelines that intersect counties but retain
# some original mileage outside the county allocation.
partially_outside = pipeline_audit.loc[
    pipeline_audit["allocation_status"]
    == "Partially outside county allocation"
]

# I report their number and total unallocated mileage.
print("\nPARTIALLY OUTSIDE COUNTY ALLOCATION")
print(f"Source records: {len(partially_outside):,}")
print(
    "Unallocated mileage:",
    f"{partially_outside['unallocated_miles'].sum():,.2f}"
)


# ---------------------------------------------------------
# I identify potential over-allocation.
# ---------------------------------------------------------

# I isolate records whose county-allocated mileage
# exceeds the original mileage beyond the tolerance.
overallocated = pipeline_audit.loc[
    pipeline_audit["allocation_status"] == "Overallocated to counties"
]

# I report the number of potentially overallocated records.
print("\nOVERALLOCATED RECORDS")
print(f"Source records: {len(overallocated):,}")


# ---------------------------------------------------------
# I verify the statewide mileage reconciliation.
# ---------------------------------------------------------

# I confirm that the difference between original and allocated
# mileage equals the sum of record-level differences.
# I allow 0.01 miles of numerical tolerance.
assert np.isclose(
    original_total - allocated_total,
    unallocated_total,
    atol=0.01
)


# ---------------------------------------------------------
# I summarize allocation differences by pipeline status.
# ---------------------------------------------------------

# I group records by original RRC status and allocation category.
# I use this to identify which pipeline statuses contribute
# to unallocated or potentially overallocated mileage.
status_audit = (
    pipeline_audit
    .groupby(["STATUS_CD", "allocation_status"], dropna=False)
    .agg(
        records=("pipeline_record_id", "size"),
        unallocated_miles=("unallocated_miles", "sum")
    )
    .reset_index()
)


# ---------------------------------------------------------
# I summarize allocation differences by system type.
# ---------------------------------------------------------

# I group records by RRC pipeline system type and allocation category.
# I use this to investigate whether particular pipeline commodities
# account for a disproportionate share of allocation differences.
system_audit = (
    pipeline_audit
    .groupby(["SYSTYPE", "allocation_status"], dropna=False)
    .agg(
        records=("pipeline_record_id", "size"),
        unallocated_miles=("unallocated_miles", "sum")
    )
    .reset_index()
)


# ---------------------------------------------------------
# I save the pipeline validation tables.
# ---------------------------------------------------------

# I save the complete source-record allocation audit.
# I preserve one row per original pipeline record.
pipeline_audit.to_csv(
    diagnostic_dir / "Pipeline_Source_Record_Allocation_Audit.csv",
    index=False
)

# I save the summary of geographic allocation categories.
allocation_summary.to_csv(
    diagnostic_dir / "Pipeline_Final_Allocation_Summary.csv",
    index=False
)

# I save the allocation audit grouped by RRC status.
status_audit.to_csv(
    diagnostic_dir / "Pipeline_Allocation_By_Status.csv",
    index=False
)

# I save the allocation audit grouped by pipeline system type.
system_audit.to_csv(
    diagnostic_dir / "Pipeline_Allocation_By_System.csv",
    index=False
)


# ---------------------------------------------------------
# I check whether the required UIC outputs exist.
# ---------------------------------------------------------

# I identify the UIC files expected from Notebook 02.
# I check their availability because later processing
# may depend on these administrative and spatial outputs.
uic_files = {
    "Administrative overlap audit":
        OUT / "TX_UIC_Disposal_Overlap_Audit.csv",
    "County-mapped UIC candidates":
        OUT / "TX_UIC_Disposal_Candidates_By_County.gpkg",
}

# I create a table showing whether each expected file exists.
# I also preserve its full path for troubleshooting.
uic_check = pd.DataFrame([
    {
        "dataset": name,
        "exists": path.exists(),
        "path": str(path)
    }
    for name, path in uic_files.items()
])

# I display the UIC file availability results.
print("\nUIC OUTPUT AVAILABILITY")
display(uic_check)


# ---------------------------------------------------------
# I distinguish file availability from data validation.
# ---------------------------------------------------------

# I check whether all expected UIC files are present.
if not uic_check["exists"].all():

    # I flag missing outputs for investigation.
    print(
        "REVIEW REQUIRED: One or more UIC outputs are missing."
    )

else:

    # I confirm file availability without claiming
    # that the administrative records represent unique APIs.
    # I retain unique-API deduplication as a separate validation.
    print(
        "UIC administrative outputs are available. "
        "Unique-API deduplication remains a separate validation."
    )


# ---------------------------------------------------------
# I flag potential pipeline over-allocation.
# ---------------------------------------------------------

# I check whether the audit identified overallocated records.
if not overallocated.empty:

    # I flag these records for investigation rather than
    # silently accepting potentially duplicated mileage.
    print(
        "\nREVIEW REQUIRED: Some pipeline records have "
        "more allocated mileage than original mileage."
    )

else:

    # I report that no material source-record over-allocation
    # was detected under the current classification.
    print(
        "\nNo material source-record over-allocation detected."
    )


# I display the directory containing the saved diagnostic files.
print("\nFinal diagnostic files saved to:", diagnostic_dir)